# Ottimizzazione degli Iperparametri — Medical VQA con Optuna

Questo notebook implementa la ricerca automatica degli iperparametri ottimali per il modello di Medical VQA, usando **Optuna** con il sampler **TPE** (Tree-structured Parzen Estimator). L'idea è evitare la classica grid search manuale e lasciare che un algoritmo di ottimizzazione bayesiana esplori lo spazio degli iperparametri in modo intelligente, concentrandosi progressivamente nelle regioni più promettenti.

La ricerca è strutturata in due fasi distinte:
- **Fase 1** — 50 trial con spazio ampio e 10 epoche per trial: serve a capire quali regioni dello spazio valgono la pena di esplorare
- **Fase 2** — 20 trial con spazio ristretto e 20 epoche per trial: raffina la ricerca attorno ai migliori parametri trovati nella fase precedente

La metrica obiettivo è il **BERTScore F1** sul validation set, scelta perché cattura la similarità semantica tra risposta generata e risposta di riferimento — molto più informativa dell'exact match per domande aperte in ambito medico.

Tutto il codice del modello, della loss e del training loop è identico al notebook principale `medicalvqa.ipynb`, per garantire piena coerenza tra ottimizzazione e training finale.

## Installazione delle dipendenze base

Installiamo `transformers` e `datasets` di HuggingFace, le due librerie fondamentali per caricare i modelli pre-addestrati (ViT-Large e Bio_ClinicalBERT) e per gestire il dataset in formato Arrow. Il flag `-q` sopprime l'output dell'installazione per tenere il log del notebook pulito.

In [ ]:
!pip install -q transformers datasets

## Installazione di BERTScore

BERTScore è la metrica che useremo come obiettivo dell'ottimizzazione. A differenza dell'exact match o del BLEU, valuta le risposte usando embeddings contestuali di BERT: questo la rende molto più robusta per risposte mediche che possono essere corrette anche se formulate diversamente (sinonimi, parafrasature). La installiamo separatamente perché richiede il download di un modello BERT dedicato al momento del primo utilizzo.

In [ ]:
!pip install -q bert-score

## Script completo di ottimizzazione — Pipeline Optuna

Questo è il cuore del file. Contiene tutta la logica di ricerca degli iperparametri, organizzata in sezioni numerate che ricalcano la struttura del notebook di training principale:

**Sezione 0 — Import**: PyTorch, librerie HuggingFace, BERTScore e Optuna con TPESampler e MedianPruner.

**Sezione 1 — Configurazione globale**: Percorsi del dataset (Kaggle), database SQLite per i trial, file di checkpoint del modello migliore. Qui si definiscono anche il numero di trial e le epoche per ogni fase.

**Sezioni 2–4 — Dataset, Modello, Loss**: Identiche al notebook principale per garantire piena coerenza. L'unica differenza è che `n_layers_to_unfreeze`, `num_heads` e `num_decoder_layers` diventano parametri ottimizzabili da Optuna anziché valori fissi.

**Sezione 5 — run_epoch**: Funzione di training/validation analoga al notebook originale, con l'aggiunta del parametro `accumulation_steps` per il gradient accumulation, anch'esso incluso nello spazio di ricerca.

**Sezione 6 — evaluate_bertscore**: Calcola il BERTScore F1 sul validation set usando la beam search. Include due fix importanti: sostituzione delle stringhe vuote con un placeholder neutro (che farebbe crashare BERTScorer) e cattura delle eccezioni per non bloccare l'ottimizzazione in caso di errori.

**Sezione 7 — make_objective**: Costruisce la funzione obiettivo di Optuna. In Fase 1 lo spazio di ricerca è ampio (es. lr da 1e-5 a 1e-3); in Fase 2 viene ristretto attorno alle regioni più promettenti. Ogni trial crea il modello da zero, lo addestra e restituisce il miglior F1 raggiunto. Il `MedianPruner` taglia i trial che si rivelano peggiori della mediana, risparmiando tempo di compute.

**Sezione 8 — main**: Lancia le due fasi in sequenza, stampa i risultati e, quando disponibile, calcola l'importanza relativa di ciascun iperparametro con `optuna.importance.get_param_importances`.

**Sezione 9 — load_best_model**: Utility per ricaricare il modello migliore trovato, leggendo la configurazione ottimale dal file JSON salvato durante l'ottimizzazione.

In [ ]:
"""
=============================================================================
  OPTUNA HYPERPARAMETER OPTIMIZATION — Medical VQA (BERTScore F1 Target)
=============================================================================

Strategia: TPE Sampler + MedianPruner (2 fasi)
  • Fase 1 — 50 trial, spazio ampio,   10 epoch/trial  → identifica le regioni
  • Fase 2 — 20 trial, spazio stretto, 20 epoch/trial  → rifinitura finale

Obiettivo: massimizzare BERTScore F1 sul validation set.

NOTA: Questo file è un wrapper Optuna attorno al codice di medicalvqa-v2.ipynb.
      Dataset, run_epoch, modello e loss sono identici al notebook originale.
=============================================================================
"""

# ─────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────
import gc
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertModel, ViTModel,
    AutoTokenizer, AutoImageProcessor,
)
from datasets import load_from_disk
from bert_score import BERTScorer
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import logging

# ─────────────────────────────────────────────
# 1. CONFIGURAZIONE GLOBALE
# ─────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/angelo01paldino/vqarad-final/VQA_RAD Output FolderValidation"
STUDY_DB     = "sqlite:////kaggle/working/optuna_medvqa.db"
BEST_CKPT    = "/kaggle/working/best_optuna_model.pth"
BEST_CFG     = "/kaggle/working/best_optuna_config.json"

EPOCHS_PHASE1   = 10
EPOCHS_PHASE2   = 20
N_TRIALS_PHASE1 = 50
N_TRIALS_PHASE2 = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

logging.getLogger("transformers").setLevel(logging.ERROR)


# ─────────────────────────────────────────────
# 2. DATASET  — identico a medicalvqa-v2.ipynb
# ─────────────────────────────────────────────
class HFVQARADDataset(Dataset):
    def __init__(self, hf_dataset_split, tokenizer, image_processor, max_len=60):
        self.dataset         = hf_dataset_split
        self.tokenizer       = tokenizer
        self.image_processor = image_processor
        self.max_len         = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        image             = item['image'].convert("RGB")
        original_question = str(item['question'])
        answer            = str(item['answer'])

        ans_type_str      = str(item.get('answer_type', 'CLOSED')).strip().upper()
        question_type_idx = 0 if ans_type_str == "CLOSED" else 1
        q_type_category   = str(item.get('question_type', 'UNKNOWN')).strip().upper()
        organ_category    = str(item.get('image_organ',   'UNKNOWN')).strip().upper()

        contextualized_question = f"[{organ_category} | {q_type_category}] {original_question}"

        pixel_values = self.image_processor(image, return_tensors="pt").pixel_values.squeeze(0)

        q_tokens = self.tokenizer(contextualized_question, truncation=True,
                                  padding='max_length', max_length=self.max_len,
                                  return_tensors="pt")
        a_tokens = self.tokenizer(answer, truncation=True,
                                  padding='max_length', max_length=self.max_len,
                                  return_tensors="pt")

        return {
            "pixel_values":   pixel_values,
            "input_ids":      q_tokens.input_ids.squeeze(0),
            "attention_mask": q_tokens.attention_mask.squeeze(0),
            "labels":         a_tokens.input_ids.squeeze(0),
            "question_type":  torch.tensor(question_type_idx, dtype=torch.long),
            "ans_type_str":   ans_type_str,
            "q_type_str":     q_type_category,
            "organ_str":      organ_category,
            "original_q":     original_question,
        }


# ─────────────────────────────────────────────
# 3. MODELLO — identico a medicalvqa-v2.ipynb,
#    con n_layers_to_unfreeze, num_heads e
#    num_decoder_layers parametrizzati da Optuna
# ─────────────────────────────────────────────
class CustomMedVQAModel(nn.Module):
    def __init__(self, vocab_size,
                 dropout=0.3,
                 n_layers_to_unfreeze=4,
                 num_heads=8,
                 num_decoder_layers=4):
        super().__init__()

        # Vision encoder
        self.vision_encoder = ViTModel.from_pretrained("google/vit-large-patch32-384")
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        for layer in self.vision_encoder.encoder.layer[-n_layers_to_unfreeze:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in self.vision_encoder.layernorm.parameters():
            p.requires_grad = True

        # Language embeddings
        bert_base      = BertModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.embedding = bert_base.embeddings.word_embeddings

        self.vision_projection = nn.Linear(1024, 768)
        self.cross_attention   = nn.MultiheadAttention(embed_dim=768,
                                                        num_heads=num_heads,
                                                        batch_first=True)
        self.layer_norm        = nn.LayerNorm(768)

        # Head 1 — Closed questions
        self.closed_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, vocab_size),
        )

        # Head 2 — Open questions (con tgt_mask come nel notebook originale)
        decoder_layer = nn.TransformerDecoderLayer(d_model=768,
                                                    nhead=num_heads,
                                                    batch_first=True,
                                                    dropout=dropout)
        self.decoder  = nn.TransformerDecoder(decoder_layer,
                                               num_layers=num_decoder_layers)
        self.fc_out   = nn.Linear(768, vocab_size)
        self.fc_out.weight = self.embedding.weight

    def forward(self, pixel_values, question_ids, answer_ids):
        vision_feats   = self.vision_encoder(pixel_values).last_hidden_state
        vision_feats   = self.vision_projection(vision_feats)
        question_feats = self.embedding(question_ids)

        attn_output, _ = self.cross_attention(query=vision_feats,
                                               key=question_feats,
                                               value=question_feats)
        memory = self.layer_norm(vision_feats + attn_output)

        pooled_memory = memory.mean(dim=1)
        closed_logits = self.closed_head(pooled_memory)

        answer_embeds = self.embedding(answer_ids)
        # ← tgt_mask identica al notebook originale
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            answer_ids.size(1)
        ).to(pixel_values.device)
        output      = self.decoder(tgt=answer_embeds, memory=memory, tgt_mask=tgt_mask)
        open_logits = self.fc_out(output)

        return closed_logits, open_logits

    @torch.no_grad()
    def generate(self, pixel_values, question_ids, question_types,
                 tokenizer, num_beams=5, max_len=32):
        """Beam search identico al notebook originale."""
        self.eval()
        batch_size = pixel_values.size(0)
        dev        = pixel_values.device

        vision_feats   = self.vision_encoder(pixel_values).last_hidden_state
        vision_feats   = self.vision_projection(vision_feats)
        question_feats = self.embedding(question_ids)
        attn_output, _ = self.cross_attention(query=vision_feats,
                                               key=question_feats,
                                               value=question_feats)
        memory = self.layer_norm(vision_feats + attn_output)

        pooled_memory = memory.mean(dim=1)
        closed_logits = self.closed_head(pooled_memory)
        closed_preds  = closed_logits.argmax(dim=-1)

        memory_expanded = memory.repeat_interleave(num_beams, dim=0)
        generated       = torch.full((batch_size * num_beams, 1),
                                     tokenizer.cls_token_id,
                                     dtype=torch.long).to(dev)
        beam_scores = torch.zeros((batch_size, num_beams)).to(dev)
        beam_scores[:, 1:] = -1e9
        beam_scores = beam_scores.view(-1)

        for _ in range(max_len):
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                generated.size(1)
            ).to(dev)
            output           = self.decoder(tgt=self.embedding(generated),
                                            memory=memory_expanded,
                                            tgt_mask=tgt_mask)
            next_token_logits = self.fc_out(output[:, -1, :])
            next_token_probs  = torch.log_softmax(next_token_logits, dim=-1)

            next_scores = next_token_probs + beam_scores[:, None]
            next_scores = next_scores.view(batch_size,
                                           num_beams * next_token_probs.size(-1))

            topk_scores, topk_indices = torch.topk(next_scores, num_beams, dim=1)
            beam_ids  = topk_indices // next_token_probs.size(-1)
            token_ids = topk_indices %  next_token_probs.size(-1)

            new_generated = []
            for i in range(batch_size):
                for j in range(num_beams):
                    prev_idx = i * num_beams + beam_ids[i, j]
                    new_seq  = torch.cat([generated[prev_idx],
                                          token_ids[i, j].unsqueeze(0)])
                    new_generated.append(new_seq)

            generated   = torch.stack(new_generated)
            beam_scores = topk_scores.view(-1)
            if (token_ids == tokenizer.sep_token_id).all():
                break

        best_generated = generated.view(batch_size, num_beams, -1)[:, 0, :]

        for i in range(batch_size):
            if question_types[i] == 0:
                best_generated[i]    = tokenizer.pad_token_id
                best_generated[i, 0] = tokenizer.cls_token_id
                best_generated[i, 1] = closed_preds[i]
                best_generated[i, 2] = tokenizer.sep_token_id

        return best_generated


# ─────────────────────────────────────────────
# 4. LOSS — identica a medicalvqa-v2.ipynb
# ─────────────────────────────────────────────
class MedVQAMultiClassFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, ignore_index=-100, reduction='mean'):
        super().__init__()
        self.gamma        = gamma
        self.ignore_index = ignore_index
        self.reduction    = reduction

    def forward(self, logits, targets):
        ce_loss    = F.cross_entropy(logits, targets, reduction='none',
                                     ignore_index=self.ignore_index,
                                     label_smoothing=0.1)
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# ─────────────────────────────────────────────
# 5. RUN_EPOCH — identica a medicalvqa-v2.ipynb
#    (unica modifica: accumulation_steps è
#     passato come parametro per Optuna)
# ─────────────────────────────────────────────
def run_epoch(model, dataloader, optimizer, criterion, scaler,
              device, is_train=True, accumulation_steps=4):
    model.train() if is_train else model.eval()
    running_loss = 0.0
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for step, batch in enumerate(dataloader):
            img     = batch['pixel_values'].to(device)
            q       = batch['input_ids'].to(device)
            a       = batch['labels'].to(device)
            q_types = batch['question_type'].to(device)

            with torch.amp.autocast('cuda'):
                # Teacher Forcing: a[:, :-1] come input, a[:, 1:] come target
                # → identico al notebook originale, risolve il ValueError
                closed_logits, open_logits = model(img, q, a[:, :-1])

                loss = 0.0

                closed_mask = (q_types == 0)
                open_mask   = (q_types == 1)

                if closed_mask.any():
                    target_closed = a[closed_mask, 1]
                    logits_closed = closed_logits[closed_mask]
                    loss += criterion(logits_closed, target_closed)

                if open_mask.any():
                    # a[:, :-1] → input decoder (max_len-1 token)
                    # a[:, 1:]  → target       (max_len-1 token) ✓ dimensioni ok
                    target_open = a[open_mask, 1:].contiguous().view(-1)
                    logits_open = open_logits[open_mask].contiguous().view(
                        -1, open_logits.size(-1)
                    )
                    loss += criterion(logits_open, target_open)

            if is_train:
                scaler.scale(loss).backward()
                # Gradient accumulation (opzionale, parametrizzato)
                if (step + 1) % accumulation_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)


# ─────────────────────────────────────────────
# 6. BERTSCORE — identico a medicalvqa-v2.ipynb
# ─────────────────────────────────────────────
def evaluate_bertscore(model, val_loader, tokenizer, device,
                        num_beams=5, max_gen_len=32):
    model.eval()
    preds_all, refs_all = [], []

    with torch.no_grad():
        for batch in val_loader:
            pv  = batch['pixel_values'].to(device)
            q   = batch['input_ids'].to(device)
            qt  = batch['question_type'].to(device)
            lbl = batch['labels']

            gen_ids    = model.generate(pv, q, qt, tokenizer,
                                        num_beams=num_beams,
                                        max_len=max_gen_len)
            preds_text = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            refs_text  = tokenizer.batch_decode(lbl,     skip_special_tokens=True)

            preds_all.extend([p.strip().lower() for p in preds_text])
            refs_all.extend( [r.strip().lower() for r in refs_text])

    # ── FIX 1: filtra le coppie con stringa vuota ──────────────────────────
    # bert_score crasha su stringhe vuote con versioni recenti di transformers
    # Sostituiamo le stringhe vuote con un placeholder neutro
    preds_all = [p if p else "[empty]" for p in preds_all]
    refs_all  = [r if r else "[empty]" for r in refs_all]

    # ── FIX 2: catch dell'eccezione per non bloccare Optuna ────────────────
    # Se BERTScorer fallisce per qualsiasi motivo, restituiamo 0.0
    # così il trial viene completato e Optuna continua
    try:
        scorer = BERTScorer(model_type="emilyalsentzer/Bio_ClinicalBERT",
                            num_layers=9, device=device, lang="en")
        scorer._tokenizer.model_max_length = 512
        _, _, F1 = scorer.score(preds_all, refs_all)
        result = float(F1.mean())
    except Exception as e:
        print(f"  [WARN] BERTScore fallito: {e} → restituisco 0.0")
        result = 0.0
    finally:
        # Pulizia memoria sempre garantita
        try:
            del scorer
        except:
            pass
        torch.cuda.empty_cache()

    return result

# ─────────────────────────────────────────────
# 7. OBJECTIVE OPTUNA
# ─────────────────────────────────────────────
def make_objective(hf_dataset, tokenizer, image_processor, vocab_size,
                   n_epochs, phase_name="phase1"):

    def objective(trial: optuna.Trial) -> float:

        # ── Spazio degli iperparametri ──────────────────────────────────────
        if phase_name == "phase1":
            lr                   = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
            weight_decay         = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)
            dropout              = trial.suggest_float("dropout", 0.1, 0.5)
            n_layers_to_unfreeze = trial.suggest_int("n_layers_to_unfreeze", 2, 8)
            num_heads            = trial.suggest_categorical("num_heads", [4, 8])
            num_decoder_layers   = trial.suggest_int("num_decoder_layers", 2, 6)
            batch_size           = trial.suggest_categorical("batch_size", [2, 4, 8])
            accumulation_steps   = trial.suggest_categorical("accumulation_steps", [2, 4, 8])
            sched_patience       = trial.suggest_int("sched_patience", 2, 8)
            sched_factor         = trial.suggest_float("sched_factor", 0.1, 0.7)
            focal_gamma          = trial.suggest_float("focal_gamma", 0.5, 4.0)
            num_beams            = trial.suggest_int("num_beams", 1, 7)
            max_gen_len          = trial.suggest_int("max_gen_len", 16, 48)
        else:
            # ── Aggiorna questi range dopo aver analizzato i trial della Fase 1!
            lr                   = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
            weight_decay         = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
            dropout              = trial.suggest_float("dropout", 0.15, 0.35)
            n_layers_to_unfreeze = trial.suggest_int("n_layers_to_unfreeze", 3, 6)
            num_heads            = trial.suggest_categorical("num_heads", [8])
            num_decoder_layers   = trial.suggest_int("num_decoder_layers", 3, 5)
            batch_size           = trial.suggest_categorical("batch_size", [4, 8])
            accumulation_steps   = trial.suggest_categorical("accumulation_steps", [4, 8])
            sched_patience       = trial.suggest_int("sched_patience", 2, 5)
            sched_factor         = trial.suggest_float("sched_factor", 0.2, 0.6)
            focal_gamma          = trial.suggest_float("focal_gamma", 1.0, 3.0)
            num_beams            = trial.suggest_int("num_beams", 3, 7)
            max_gen_len          = trial.suggest_int("max_gen_len", 24, 40)

        # ── DataLoaders ─────────────────────────────────────────────────────
        train_set = HFVQARADDataset(hf_dataset['train'],      tokenizer, image_processor)
        val_set   = HFVQARADDataset(hf_dataset['validation'], tokenizer, image_processor)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                                  num_workers=2, pin_memory=True)

        # ── Modello ─────────────────────────────────────────────────────────
        model = CustomMedVQAModel(
            vocab_size=vocab_size,
            dropout=dropout,
            n_layers_to_unfreeze=n_layers_to_unfreeze,
            num_heads=num_heads,
            num_decoder_layers=num_decoder_layers,
        ).to(device)

        # ── Optimizer + Scheduler + Loss ────────────────────────────────────
        # Identici a medicalvqa-v2.ipynb, con iperparametri da Optuna
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.AdamW(trainable_params,
                                      lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=sched_factor, patience=sched_patience
        )
        criterion = MedVQAMultiClassFocalLoss(
            gamma=focal_gamma,
            ignore_index=tokenizer.pad_token_id   # ← come nel notebook originale
        ).to(device)
        scaler = torch.amp.GradScaler('cuda')

        # ── Training loop ───────────────────────────────────────────────────
        best_f1 = 0.0

        for epoch in range(n_epochs):
            train_loss = run_epoch(model, train_loader, optimizer, criterion,
                                   scaler, device, is_train=True,
                                   accumulation_steps=accumulation_steps)
            val_loss   = run_epoch(model, val_loader, None, criterion,
                                   None, device, is_train=False,
                                   accumulation_steps=accumulation_steps)
            scheduler.step(val_loss)

            # BERTScore ogni 2 epoch (costoso)
            if (epoch + 1) % 2 == 0 or epoch == n_epochs - 1:
                f1 = evaluate_bertscore(model, val_loader, tokenizer, device,
                                        num_beams=num_beams,
                                        max_gen_len=max_gen_len)
                best_f1 = max(best_f1, f1)

                trial.report(f1, epoch)
                if trial.should_prune():
                    del model
                    torch.cuda.empty_cache()
                    gc.collect()
                    raise optuna.exceptions.TrialPruned()

            print(f"  [Trial {trial.number} | {phase_name} | Epoch {epoch+1:02d}] "
                  f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"best_f1={best_f1:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")

        # ── Salva il miglior modello globale ────────────────────────────────
        if best_f1 > getattr(objective, "_global_best_f1", 0.0):
            objective._global_best_f1 = best_f1
            torch.save(model.state_dict(), BEST_CKPT)
            cfg = dict(trial.params)
            with open(BEST_CFG, "w") as f:
                json.dump(cfg, f, indent=2)
            print(f"  ★ Nuovo best F1={best_f1:.4f} → checkpoint salvato.")

        del model
        torch.cuda.empty_cache()
        gc.collect()
        return best_f1

    objective._global_best_f1 = 0.0
    return objective


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────
def main():
    print("Caricamento dataset...")
    hf_dataset      = load_from_disk(DATASET_PATH)
    tokenizer       = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    image_processor = AutoImageProcessor.from_pretrained("google/vit-large-patch32-384")
    vocab_size      = len(tokenizer)   # ← identico a medicalvqa-v2.ipynb (len, non .vocab_size)

    print(f"Split: {list(hf_dataset.keys())}")
    print(f"Vocab size: {vocab_size}")

    # ══════════════════════════════════════════
    #  FASE 1 — Esplorazione larga (50 trial)
    # ══════════════════════════════════════════
    print("\n" + "="*60)
    print("  FASE 1: Ricerca Larga — 50 trial, 10 epoch/trial")
    print("="*60)

    study_phase1 = optuna.create_study(
        study_name    ="medvqa_phase1",
        direction     ="maximize",
        sampler       =TPESampler(seed=42),
        pruner        =MedianPruner(n_startup_trials=5, n_warmup_steps=4, interval_steps=1),
        storage       =STUDY_DB,
        load_if_exists=True,
    )
    objective_p1 = make_objective(
        hf_dataset, tokenizer, image_processor, vocab_size,
        n_epochs=EPOCHS_PHASE1, phase_name="phase1",
    )
    study_phase1.optimize(objective_p1, n_trials=N_TRIALS_PHASE1,
                          gc_after_trial=True, show_progress_bar=True)

    print("\n── Risultati Fase 1 ──")
    print(f"  Migliore BERTScore F1 : {study_phase1.best_value:.4f}")
    print(f"  Parametri ottimali    : {study_phase1.best_params}")

    try:
        importances = optuna.importance.get_param_importances(study_phase1)
        print("\n  Importanza iperparametri (Fase 1):")
        for name, imp in sorted(importances.items(), key=lambda x: -x[1]):
            print(f"    {name:30s}: {imp:.4f}")
    except Exception as e:
        print(f"  (importanze non disponibili: {e})")

    # ══════════════════════════════════════════
    #  FASE 2 — Raffinamento (20 trial)
    #  IMPORTANTE: aggiorna i range "phase2" in
    #  make_objective() prima di lanciare!
    # ══════════════════════════════════════════
    print("\n" + "="*60)
    print("  FASE 2: Raffinamento — 20 trial, 20 epoch/trial")
    print("="*60)

    study_phase2 = optuna.create_study(
        study_name    ="medvqa_phase2",
        direction     ="maximize",
        sampler       =TPESampler(seed=42, multivariate=True,
                                  warn_independent_sampling=False),
        pruner        =MedianPruner(n_startup_trials=3, n_warmup_steps=6, interval_steps=2),
        storage       =STUDY_DB,
        load_if_exists=True,
    )
    objective_p2 = make_objective(
        hf_dataset, tokenizer, image_processor, vocab_size,
        n_epochs=EPOCHS_PHASE2, phase_name="phase2",
    )
    study_phase2.optimize(objective_p2, n_trials=N_TRIALS_PHASE2,
                          gc_after_trial=True, show_progress_bar=True)

    print("\n" + "="*60)
    print("  RISULTATI FINALI")
    print("="*60)
    print(f"  Fase 1 — Best BERTScore F1 : {study_phase1.best_value:.4f}")
    print(f"  Fase 2 — Best BERTScore F1 : {study_phase2.best_value:.4f}")
    print(f"\n  Configurazione ottimale (Fase 2):")
    for k, v in study_phase2.best_params.items():
        print(f"    {k:30s}: {v}")
    print(f"\n  Checkpoint migliore : {BEST_CKPT}")
    print(f"  Config migliore     : {BEST_CFG}")
    print(f"  DB Optuna           : {STUDY_DB}")

    return study_phase1, study_phase2


# ─────────────────────────────────────────────
# 9. UTILITY — Carica il miglior modello
# ─────────────────────────────────────────────
def load_best_model(tokenizer, device):
    with open(BEST_CFG) as f:
        cfg = json.load(f)
    print("Configurazione caricata:")
    for k, v in cfg.items():
        print(f"  {k}: {v}")

    model = CustomMedVQAModel(
        vocab_size           = len(tokenizer),
        dropout              = cfg["dropout"],
        n_layers_to_unfreeze = cfg["n_layers_to_unfreeze"],
        num_heads            = cfg["num_heads"],
        num_decoder_layers   = cfg["num_decoder_layers"],
    ).to(device)

    model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
    model.eval()
    print(f"Modello caricato da {BEST_CKPT}")
    return model, cfg


# ─────────────────────────────────────────────
# 10. ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    study1, study2 = main()